In [3]:
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
import os
import json
import numpy as np
import google.generativeai as genai

from groq import Groq
from sklearn.metrics.pairwise import cosine_similarity


class TextEvaluator:
    """
    Pure Text-Based Evaluation

    No:
    - RAG
    - Retrieval
    - Context documents
    - Vector DB
    - External grounding

    Everything operates directly on text inputs.
    """

    def __init__(self, model_name: str = "llama-3.3-70b-versatile"):

        self.client = Groq(
            api_key=os.getenv("GROQ_API_KEY")
        )

        self.model = model_name

        genai.configure(
            api_key=os.getenv("GEMINI_API_KEY")
        )

        self.embed_model = "gemini-embedding-001"

    # ==========================================================
    # LLM CALL
    # ==========================================================

    def _llm_call(self, prompt: str) -> str:

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
        )

        return response.choices[0].message.content.strip()

    # ==========================================================
    # JSON PARSER
    # ==========================================================

    def _parse_json(self, text: str):

        try:

            start_idx = min([
                text.find(b)
                for b in ['[', '{']
                if text.find(b) != -1
            ])

            end_idx = max([
                text.rfind(b)
                for b in [']', '}']
                if text.rfind(b) != -1
            ])

            return json.loads(
                text[start_idx:end_idx + 1]
            )

        except:
            return None

    # ==========================================================
    # ANSWER GENERATION
    # ==========================================================

    def generate_answer(self, question: str) -> str:

        prompt = f"""
        Answer the following question clearly and accurately.

        Question:
        {question}
        """

        return self._llm_call(prompt)

    # ==========================================================
    # FAITHFULNESS (TEXT vs TEXT)
    # ==========================================================

    def calculate_faithfulness(
        self,
        reference_text: str,
        generated_text: str
    ) -> dict:

        # Step 1: Extract claims
        extract_prompt = f"""
        Extract factual statements from the following text.

        Text:
        {generated_text}

        Return ONLY a JSON list of strings.
        """

        statements = self._parse_json(
            self._llm_call(extract_prompt)
        )

        if not statements:
            return {"score": 0.0}

        # Step 2: Verify against reference text
        verify_prompt = f"""
        Determine whether each statement is supported
        by the reference text.

        Reference Text:
        {reference_text}

        Statements:
        {json.dumps(statements)}

        Return ONLY a JSON list like:
        [
            {{
                "statement": "...",
                "verdict": "Yes/No",
                "explanation": "..."
            }}
        ]
        """

        verdicts = self._parse_json(
            self._llm_call(verify_prompt)
        )

        if not verdicts:
            return {"score": 0.0}

        supported = sum(
            1 for v in verdicts
            if v.get("verdict") == "Yes"
        )

        score = supported / len(statements)

        return {
            "score": float(score),
            "details": verdicts
        }

    # ==========================================================
    # ANSWER RELEVANCY
    # ==========================================================

    def calculate_answer_relevancy(
        self,
        question: str,
        answer: str,
        n: int = 3
    ) -> dict:

        generated_questions = []

        # Reverse question generation
        for _ in range(n):

            prompt = f"""
            Generate a question for the given answer.

            Answer:
            {answer}
            """

            generated_questions.append(
                self._llm_call(prompt)
            )

        # Original question embedding
        original_embedding_response = genai.embed_content(
            model=self.embed_model,
            content=question,
            task_type="retrieval_query"
        )

        original_embedding = np.array(
            original_embedding_response["embedding"]
        ).reshape(1, -1)

        # Generated question embeddings
        generated_embeddings = []

        for q in generated_questions:

            embedding_response = genai.embed_content(
                model=self.embed_model,
                content=q,
                task_type="retrieval_query"
            )

            generated_embeddings.append(
                np.array(
                    embedding_response["embedding"]
                )
            )

        similarities = cosine_similarity(
            original_embedding,
            generated_embeddings
        )[0]

        score = float(np.mean(similarities))

        return {
            "score": score,
            "generated_questions": generated_questions,
            "similarities": similarities.tolist()
        }

    # ==========================================================
    # TEXT RELEVANCY
    # ==========================================================

    def calculate_text_relevancy(
        self,
        query_text: str,
        candidate_text: str
    ) -> dict:

        # Embed query
        query_embedding_response = genai.embed_content(
            model=self.embed_model,
            content=query_text,
            task_type="retrieval_query"
        )

        query_embedding = np.array(
            query_embedding_response["embedding"]
        ).reshape(1, -1)

        # Embed candidate text
        candidate_embedding_response = genai.embed_content(
            model=self.embed_model,
            content=candidate_text,
            task_type="retrieval_document"
        )

        candidate_embedding = np.array(
            candidate_embedding_response["embedding"]
        ).reshape(1, -1)

        similarity = cosine_similarity(
            query_embedding,
            candidate_embedding
        )[0][0]

        return {
            "score": float(similarity)
        }



In [6]:
try:
    with open("paper_tests.json", "r") as f:
        tests = json.load(f)

except FileNotFoundError:
    print("paper_tests.json not found.")
    tests = []


for test in tests:

    print(f"\n=== Testing {test['metric']} ===")

    # ==========================================================
    # FAITHFULNESS
    # ==========================================================

    if test["metric"] == "Faithfulness":

        high_score = evaluator.calculate_faithfulness(
            reference_text=test["ground_truth"],
            generated_text=test["high_answer"]
        )

        low_score = evaluator.calculate_faithfulness(
            reference_text=test["ground_truth"],
            generated_text=test["low_answer"]
        )

        print(
            f"High Faithfulness: {high_score['score']:.2f}"
        )

        print(
            f"Low Faithfulness : {low_score['score']:.2f}"
        )

    # ==========================================================
    # ANSWER RELEVANCY
    # ==========================================================

    elif test["metric"] == "Answer Relevancy":

        high_score = evaluator.calculate_answer_relevancy(
            question=test["question"],
            answer=test["high_answer"]
        )

        low_score = evaluator.calculate_answer_relevancy(
            question=test["question"],
            answer=test["low_answer"]
        )

        print(
            f"High Relevancy: {high_score['score']:.2f}"
        )

        print(
            f"Low Relevancy : {low_score['score']:.2f}"
        )

    # ==========================================================
    # TEXT RELEVANCY
    # ==========================================================

    elif test["metric"] == "Context Relevancy":

        high_score = evaluator.calculate_text_relevancy(
            query_text=test["question"],
            candidate_text=test["high_context"]
        )

        low_score = evaluator.calculate_text_relevancy(
            query_text=test["question"],
            candidate_text=test["low_context"]
        )

        print(
            f"High Text Relevancy: {high_score['score']:.2f}"
        )

        print(
            f"Low Text Relevancy : {low_score['score']:.2f}"
        )

    print("-" * 60)


=== Testing Faithfulness ===
High Faithfulness: 1.00
Low Faithfulness : 1.00
------------------------------------------------------------

=== Testing Answer Relevancy ===
High Relevancy: 0.96
Low Relevancy : 0.86
------------------------------------------------------------

=== Testing Context Relevancy ===
High Text Relevancy: 0.79
Low Text Relevancy : 0.78
------------------------------------------------------------

=== Testing Answer Correctness ===
------------------------------------------------------------
